# ACL C++ 算子调用：SpMV

按顺序运行下面的代码，需要修改的参数已在代码中标出。


<a id="s-04-04_acl_sparse_spmv"></a>


## 4.4 ops-sparse SpMV

### 本节学习目标

- 使用 CSR descriptor 调用稀疏库
- 区分 handle/descriptor 与 ACLNN executor


### 准备 ops-sparse

本章使用 `e0015bb76090487165f43dbaf2aae37648887a4b` 版本，在 CANNLab A3 上构建 SpMV 并安装到 `~/Ascend/cann`。

首次使用时依次执行下面两格。已安装时，从“设置库路径”继续；下载完成但构建失败时，只重跑第二格。

本次构建出现 `GM_ADDR` 重定义告警，但构建和安装均已完成。阅读结果时重点看目标 SoC、打包成功和安装成功信息。

离线机房由教师提前构建并安装 `build_out/cann-A3-ops-sparse-1.0.0_linux-aarch64.run`。首次构建还会下载 makeself，因此应在联网环境准备。


In [ ]:
%%bash
set -e
git clone https://gitcode.com/cann/ops-sparse.git ops-sparse
git -C ops-sparse checkout e0015bb76090487165f43dbaf2aae37648887a4b


In [ ]:
%%bash
set -e
source "$ASCEND_HOME_PATH/set_env.sh"
cd ops-sparse
bash build.sh --pkg --soc=ascend910_93 --ops=spmv
bash build_out/cann-A3-ops-sparse-1.0.0_linux-aarch64.run --install --quiet --install-path="$HOME/Ascend"


### 设置库路径

下面设置本 Notebook 使用的头文件和动态库目录。自定义安装时，安装命令中的 `--install-path` 和这里的路径需对应；`OPS_SPARSE_ROOT` 末尾包含 `/cann`。


In [ ]:
import os

# 与安装步骤中的目录一致；自定义安装时修改这里。
os.environ["OPS_SPARSE_ROOT"] = os.path.expanduser("~/Ascend/cann")
os.environ["LD_LIBRARY_PATH"] = os.environ["OPS_SPARSE_ROOT"] + "/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")


In [ ]:
%%bash
set -e
npu-smi info


### 稀疏对象

工程创建 `aclsparseHandle_t` 并绑定 stream，使用 `aclsparseCreateCsr` 描述 CSR，使用 `aclsparseCreateDnVec` 描述 x/y。


### 执行与验证

`aclsparseSpMV` 传入 alpha、beta、operation、matrix、vectors 和数据类型；同步后回读 y，并与 Host CSR reference 比较相对误差。当前接口没有 ACLNN executor/workspace。


### 编译并运行 SpMV

下面用 Device 0 计算 SpMV，矩阵有 4096 行、4096 列和 65536 个非零元。检查相对误差是否小于 1e-6。


In [ ]:
%%bash
set -e
cd src/acl_operator_calls/SpMV-acl
cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DACL_C_STUB=OFF
cmake --build build -j
# 学生修改：--rows、--cols、--nnz 控制矩阵大小和非零元数量。
./build/bin/spmv_acl --rows 4096 --cols 4096 --nnz 65536 --warmup 3 --repeat 10 --device 0


### 预期现象与结果分析

真实环境应输出 rows/cols/nnz、ACL SpMV 时间和误差。若 CMake 报缺少 `cann_ops_sparse.h` 或 `libops_sparse.so`，回到本节开头，检查安装目录并运行安装单元格。


本章首先学习算子调用流程。这里的输入、构建方式和计时范围与第 02 章不同，不能把两章时间直接相除作为 NPU 加速比。


### 课后实践

画出 handle、CSR descriptor、dense vector descriptor 与 Device buffer 的关系。

参考答案见 `answer/04.04_answer.md`。


<a id="s-04-05_dense_sparse_api_comparison"></a>


## 4.5 稠密与稀疏 API 对比

### 本节学习目标

- 比较 dense/sparse 计算模式
- 总结两条 API 的共性与差异


### 计算模式

GEMM 规则访问两个稠密矩阵，计算密度高；SpMV 只访问非零元，但需要读取索引并间接访问 x，通常更受内存和不规则访问限制。


### API 对比

共性包括 ACL runtime、Device memory、Host/Device copy、stream、同步、reference 和清理。GEMM 特有 aclTensor + workspace/executor；SpMV 特有 sparse handle + CSR/dense-vector descriptors。


### 比较时看什么

GEMM 用 Tensor 描述矩阵，通过 workspace/executor 准备并执行计算；SpMV 用 handle 和稀疏描述符组织数据，直接调用 `aclsparseSpMV`。对照各自源码，分别列出创建、执行和释放顺序。


### 课后实践

制作一张 GEMM/SpMV 的资源、准备、执行和释放对照表。

参考答案见 `answer/04.05_answer.md`。
